# In class activity 4/23
## Feature selection
### Dirks Wright

In [1]:
### import libraries

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold, f_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

RANDOM_STATE = 42
N_JOBS = 1  # Change to -1 if your machine allows parallel sklearn jobs.
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## Load and Clean Data

In [2]:
### load dataset
adult = pd.read_csv('adult.csv').replace('?', np.nan)
adult['income'] = adult['income'].map({'<=50K': 0, '>50K': 1})
### specify X and y, split into train and test sets, and identify numeric and categorical columns
X = adult.drop(columns=['income', 'fnlwgt'])
y = adult['income']

numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Rows:', adult.shape[0])
print('Original modeling features:', X.shape[1])
print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)
display(y.value_counts(normalize=True).rename('target share'))

Rows: 48842
Original modeling features: 13
Numeric columns: ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical columns: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']


income
0    0.760718
1    0.239282
Name: target share, dtype: float64

## Engineer a Large Feature Set

In [3]:
### feature cleaning function to ensure valid column names and uniqueness after transformations
def clean_feature_names(cols):
    cleaned = pd.Index(cols).str.replace(r'[^0-9a-zA-Z_]+', '_', regex=True).str.strip('_')
    seen = {}
    output = []
    for col in cleaned:
        base = col if col else 'feature'
        count = seen.get(base, 0)
        output.append(base if count == 0 else f'{base}_{count}')
        seen[base] = count + 1
    return output

In [4]:
### function to create a large feature set with numeric transformations, categorical encoding, and interactions
def make_large_feature_set(X_train, X_test, numeric_cols, categorical_cols):
    train_num = X_train[numeric_cols].copy()
    test_num = X_test[numeric_cols].copy()
    medians = train_num.median()
    train_num = train_num.fillna(medians)
    test_num = test_num.fillna(medians)

    scaler = StandardScaler()
    train_num_z = pd.DataFrame(
        scaler.fit_transform(train_num),
        columns=[f'{col}_z' for col in numeric_cols],
        index=X_train.index,
    )
    test_num_z = pd.DataFrame(
        scaler.transform(test_num),
        columns=train_num_z.columns,
        index=X_test.index,
    )

    poly = PolynomialFeatures(degree=2, include_bias=False)
    train_poly = pd.DataFrame(
        poly.fit_transform(train_num_z),
        columns=poly.get_feature_names_out(train_num_z.columns),
        index=X_train.index,
    )
    test_poly = pd.DataFrame(
        poly.transform(test_num_z),
        columns=train_poly.columns,
        index=X_test.index,
    )

    train_cat = X_train[categorical_cols].astype('object').fillna('Missing')
    test_cat = X_test[categorical_cols].astype('object').fillna('Missing')
    train_cat_dummies = pd.get_dummies(train_cat, prefix=categorical_cols, dtype=float)
    test_cat_dummies = pd.get_dummies(test_cat, prefix=categorical_cols, dtype=float)
    test_cat_dummies = test_cat_dummies.reindex(columns=train_cat_dummies.columns, fill_value=0.0)

    train_interactions = []
    test_interactions = []
    for col in train_num_z.columns:
        train_block = train_cat_dummies.mul(train_num_z[col], axis=0)
        test_block = test_cat_dummies.mul(test_num_z[col], axis=0)
        train_block.columns = [f'{col}_x_{dummy}' for dummy in train_cat_dummies.columns]
        test_block.columns = train_block.columns
        train_interactions.append(train_block)
        test_interactions.append(test_block)

    X_train_large = pd.concat([train_poly, train_cat_dummies] + train_interactions, axis=1)
    X_test_large = pd.concat([test_poly, test_cat_dummies] + test_interactions, axis=1)
    feature_names = clean_feature_names(X_train_large.columns)
    X_train_large.columns = feature_names
    X_test_large.columns = feature_names
    return X_train_large.astype('float32'), X_test_large.astype('float32')

In [5]:
### create large feature set and drop constant features
X_train_large, X_test_large = make_large_feature_set(
    X_train_raw,
    X_test_raw,
    numeric_cols,
    categorical_cols,
)

constant_dropper = VarianceThreshold(threshold=0.0)
constant_dropper.fit(X_train_large)
kept_features = X_train_large.columns[constant_dropper.get_support()].tolist()

X_train_rank = X_train_large[kept_features]
X_test_rank = X_test_large[kept_features]

print('Engineered training shape:', X_train_large.shape)
print('Features after dropping constants:', X_train_rank.shape[1])
print('Dropped constant features:', X_train_large.shape[1] - X_train_rank.shape[1])

Engineered training shape: (39073, 632)
Features after dropping constants: 632
Dropped constant features: 0


## Rank and Reduce Features

I use three different metrics so the final feature set is not driven by one model assumption:

- ANOVA F scores for univariate separation.
- L1 logistic regression coefficients for sparse linear signal.
- Extra Trees feature importances for nonlinear/tree-based signal.

In [6]:
### compute feature importance scores from ANOVA F-test, L1 logistic regression, and tree-based model, then rank and select top features
anova_scores, _ = f_classif(X_train_rank, y_train)
anova_scores = np.nan_to_num(anova_scores, nan=0.0, posinf=0.0, neginf=0.0)

l1_selector = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.1,
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE,
)
l1_selector.fit(X_train_rank, y_train)
l1_scores = np.abs(l1_selector.coef_).ravel()

tree_selector = ExtraTreesClassifier(
    n_estimators=80,
    class_weight='balanced',
    max_features='sqrt',
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)
tree_selector.fit(X_train_rank, y_train)

rank_df = pd.DataFrame({
    'feature': kept_features,
    'anova_score': anova_scores,
    'l1_abs_coef': l1_scores,
    'tree_importance': tree_selector.feature_importances_,
})

for score_col in ['anova_score', 'l1_abs_coef', 'tree_importance']:
    rank_df[f'{score_col}_rank'] = rank_df[score_col].rank(ascending=False, method='min')

rank_cols = ['anova_score_rank', 'l1_abs_coef_rank', 'tree_importance_rank']
rank_df['average_rank'] = rank_df[rank_cols].mean(axis=1)
rank_df = rank_df.sort_values(['average_rank', 'tree_importance'], ascending=[True, False]).reset_index(drop=True)

selected_features = rank_df.head(20)['feature'].tolist()
X_train_20 = X_train_rank[selected_features]
X_test_20 = X_test_rank[selected_features]

display(rank_df.head(25))
print('Selected feature count:', len(selected_features))
print('\nSelected features:')
for feature in selected_features:
    print('-', feature)

,feature,anova_score,l1_abs_coef,tree_importance,anova_score_rank,l1_abs_coef_rank,tree_importance_rank,average_rank
0,marital_status_Married_civ_spouse,9876.470224,1.747841,0.054086,1.0,2.0,1.0,1.333333
1,marital_status_Never_married,4504.460151,0.448683,0.027807,6.0,17.0,3.0,8.666667
2,age_z,2197.103159,0.590656,0.017278,14.0,12.0,4.0,10.000000
3,relationship_Own_child,2070.663244,0.374313,0.016087,16.0,22.0,5.0,14.333333
4,hours_per_week_z,2079.889689,0.638398,0.011531,15.0,8.0,20.0,14.333333
5,gender_Female,1893.413026,0.722107,0.010574,23.0,6.0,21.0,16.666667
6,educational_num_z,4840.925682,0.375861,0.008849,4.0,21.0,27.0,17.333333
7,capital_gain_z,2010.757849,2.177064,0.007458,19.0,1.0,34.0,18.000000
8,occupation_Exec_managerial,1894.686387,0.651956,0.009584,22.0,7.0,26.0,18.333333
9,age_z_x_marital_status_Never_married,2741.321925,0.340255,0.010561,12.0,26.0,22.0,20.000000


Selected feature count: 20

Selected features:
- marital_status_Married_civ_spouse
- marital_status_Never_married
- age_z
- relationship_Own_child
- hours_per_week_z
- gender_Female
- educational_num_z
- capital_gain_z
- occupation_Exec_managerial
- age_z_x_marital_status_Never_married
- age_z_x_gender_Male
- occupation_Prof_specialty
- occupation_Other_service
- educational_num_z_x_native_country_United_States
- educational_num_z_x_workclass_Private
- age_z_x_native_country_United_States
- relationship_Wife
- hours_per_week_z_x_marital_status_Married_civ_spouse
- relationship_Unmarried
- age_z_x_marital_status_Married_civ_spouse


In [7]:
### removed features reasons 
def removal_reason(feature):
    if '_x_' in feature:
        return 'engineered interaction ranked below the final top 20'
    if 'native_country' in feature:
        return 'country dummy or country interaction had weaker or rarer signal'
    if feature.startswith('education_'):
        return 'education dummy ranked below educational_num or stronger indicators'
    return 'combined ANOVA, L1, and tree ranking placed it below the selected features'


removed_features = [feature for feature in X_train_rank.columns if feature not in selected_features]
removed_examples = pd.DataFrame({
    'removed_feature': ['fnlwgt'] + removed_features[:25],
    'reason': ['sampling weight, not a direct person-level predictor'] + [removal_reason(feature) for feature in removed_features[:25]],
})

print('Removed feature count, including fnlwgt:', len(removed_features) + 1)
display(removed_examples)

Removed feature count, including fnlwgt: 613


,removed_feature,reason
0,fnlwgt,"sampling weight, not a direct person-level pre..."
1,capital_loss_z,"combined ANOVA, L1, and tree ranking placed it..."
2,age_z_2,"combined ANOVA, L1, and tree ranking placed it..."
3,age_z_educational_num_z,"combined ANOVA, L1, and tree ranking placed it..."
4,age_z_capital_gain_z,"combined ANOVA, L1, and tree ranking placed it..."
5,age_z_capital_loss_z,"combined ANOVA, L1, and tree ranking placed it..."
6,age_z_hours_per_week_z,"combined ANOVA, L1, and tree ranking placed it..."
7,educational_num_z_2,"combined ANOVA, L1, and tree ranking placed it..."
8,educational_num_z_capital_gain_z,"combined ANOVA, L1, and tree ranking placed it..."
9,educational_num_z_capital_loss_z,"combined ANOVA, L1, and tree ranking placed it..."


## Performance Before and After Feature Reduction

In [8]:
### Baseline model evaluation on different feature sets
def evaluate_model(name, model, X_fit, y_fit, X_eval, y_eval):
    fitted = clone(model).fit(X_fit, y_fit)
    pred = fitted.predict(X_eval)
    proba = fitted.predict_proba(X_eval)[:, 1]
    return {
        'model': name,
        'balanced_accuracy': balanced_accuracy_score(y_eval, pred),
        'accuracy': accuracy_score(y_eval, pred),
        'f1': f1_score(y_eval, pred),
        'roc_auc': roc_auc_score(y_eval, proba),
    }, fitted


logit_baseline = LogisticRegression(
    solver='liblinear',
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE,
)

rf_baseline = RandomForestClassifier(
    n_estimators=120,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)

feature_set_rows = []
for label, X_fit, X_eval in [
    ('large engineered set', X_train_rank, X_test_rank),
    ('selected 20 features', X_train_20, X_test_20),
]:
    for model_name, model in [
        ('Logistic baseline', logit_baseline),
        ('Random Forest baseline', rf_baseline),
    ]:
        row, _ = evaluate_model(f'{model_name} on {label}', model, X_fit, y_train, X_eval, y_test)
        feature_set_rows.append(row)

feature_set_comparison = pd.DataFrame(feature_set_rows)
display(feature_set_comparison.sort_values('balanced_accuracy', ascending=False))

,model,balanced_accuracy,accuracy,f1,roc_auc
0,Logistic baseline on large engineered set,0.824324,0.810114,0.682200,0.909945
2,Logistic baseline on selected 20 features,0.812518,0.791483,0.661909,0.897954
1,Random Forest baseline on large engineered set,0.781458,0.851264,0.675742,0.899188
3,Random Forest baseline on selected 20 features,0.777792,0.826287,0.653603,0.884626


## Tune Two Different Base Models

In [9]:
### hyperparameter tuning for logistic regression and random forest on selected features
logit_grid = GridSearchCV(
    LogisticRegression(
        solver='liblinear',
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    param_grid={
        'penalty': ['l1', 'l2'],
        'C': [0.03, 0.1, 0.3, 1, 3, 10],
    },
    scoring='balanced_accuracy',
    cv=CV,
    n_jobs=N_JOBS,
    refit=True,
)
logit_grid.fit(X_train_20, y_train)

rf_search = RandomizedSearchCV(
    RandomForestClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
    param_distributions={
        'n_estimators': [120, 200, 300],
        'max_depth': [None, 8, 14, 24],
        'min_samples_leaf': [1, 3, 8, 15],
        'min_samples_split': [2, 5, 10],
        'max_features': ['sqrt', 0.5, None],
    },
    n_iter=8,
    scoring='balanced_accuracy',
    cv=CV,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    refit=True,
)
rf_search.fit(X_train_20, y_train)

print('Best logistic parameters:', logit_grid.best_params_)
print('Best logistic CV balanced accuracy:', round(logit_grid.best_score_, 4))
print('Best random forest parameters:', rf_search.best_params_)
print('Best random forest CV balanced accuracy:', round(rf_search.best_score_, 4))

Best logistic parameters: {'C': 3, 'penalty': 'l1'}
Best logistic CV balanced accuracy: 0.817
Best random forest parameters: {'n_estimators': 120, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 8}
Best random forest CV balanced accuracy: 0.8262


In [10]:
### final ecaluation of tuned models on test set
base_results = []
for name, estimator, cv_score in [
    ('Tuned Logistic Regression', logit_grid.best_estimator_, logit_grid.best_score_),
    ('Tuned Random Forest', rf_search.best_estimator_, rf_search.best_score_),
]:
    row, _ = evaluate_model(name, estimator, X_train_20, y_train, X_test_20, y_test)
    row['best_cv_balanced_accuracy'] = cv_score
    base_results.append(row)

base_results = pd.DataFrame(base_results)
display(base_results.sort_values('balanced_accuracy', ascending=False))

,model,balanced_accuracy,accuracy,f1,roc_auc,best_cv_balanced_accuracy
1,Tuned Random Forest,0.822606,0.801924,0.675716,0.910394,0.826228
0,Tuned Logistic Regression,0.812451,0.791381,0.661799,0.897959,0.816971


## Permutation Importance on the Reduced Features

In [11]:
### permutation importance
perm_frames = []
for name, model in [
    ('logistic', logit_grid.best_estimator_),
    ('random_forest', rf_search.best_estimator_),
]:
    perm = permutation_importance(
        model,
        X_test_20,
        y_test,
        scoring='balanced_accuracy',
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )
    perm_frames.append(pd.DataFrame({
        'feature': selected_features,
        f'{name}_perm': perm.importances_mean,
    }))

perm_summary = perm_frames[0].merge(perm_frames[1], on='feature')
perm_summary['mean_perm'] = perm_summary[['logistic_perm', 'random_forest_perm']].mean(axis=1)
perm_summary = perm_summary.sort_values('mean_perm', ascending=False).reset_index(drop=True)
display(perm_summary)

,feature,logistic_perm,random_forest_perm,mean_perm
0,marital_status_Married_civ_spouse,0.074954,0.145889,0.110422
1,educational_num_z,0.020816,0.049553,0.035184
2,capital_gain_z,0.023321,0.031975,0.027648
3,hours_per_week_z,0.028244,0.006667,0.017456
4,gender_Female,0.012980,0.000120,0.006550
5,age_z,0.006071,0.005566,0.005818
6,occupation_Other_service,0.005931,0.002048,0.003990
7,age_z_x_marital_status_Never_married,0.006579,0.001226,0.003903
8,relationship_Own_child,0.005269,0.000077,0.002673
9,relationship_Wife,0.005391,-0.000161,0.002615


## OOF Stacking Meta Learner

In [12]:
### stacking with out-of-fold predictions from tuned base models
base_models = [
    ('logistic', logit_grid.best_estimator_),
    ('random_forest', rf_search.best_estimator_),
]

oof_train = np.zeros((X_train_20.shape[0], len(base_models)))
test_meta_folds = np.zeros((X_test_20.shape[0], len(base_models), CV.get_n_splits()))

for fold, (tr_idx, val_idx) in enumerate(CV.split(X_train_20, y_train)):
    X_tr, X_val = X_train_20.iloc[tr_idx], X_train_20.iloc[val_idx]
    y_tr = y_train.iloc[tr_idx]

    for model_idx, (name, model) in enumerate(base_models):
        fitted = clone(model).fit(X_tr, y_tr)
        oof_train[val_idx, model_idx] = fitted.predict_proba(X_val)[:, 1]
        test_meta_folds[:, model_idx, fold] = fitted.predict_proba(X_test_20)[:, 1]

test_meta = test_meta_folds.mean(axis=2)

meta_search = GridSearchCV(
    LogisticRegression(
        solver='lbfgs',
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    param_grid={'C': [0.1, 0.3, 1, 3, 10]},
    scoring='balanced_accuracy',
    cv=CV,
    n_jobs=N_JOBS,
    refit=True,
)
meta_search.fit(oof_train, y_train)

stack_pred = meta_search.predict(test_meta)
stack_proba = meta_search.predict_proba(test_meta)[:, 1]
stack_row = {
    'model': 'OOF stacked model',
    'balanced_accuracy': balanced_accuracy_score(y_test, stack_pred),
    'accuracy': accuracy_score(y_test, stack_pred),
    'f1': f1_score(y_test, stack_pred),
    'roc_auc': roc_auc_score(y_test, stack_proba),
    'best_cv_balanced_accuracy': meta_search.best_score_,
}

final_results = pd.concat([base_results, pd.DataFrame([stack_row])], ignore_index=True)

print('Meta learner best parameters:', meta_search.best_params_)
display(final_results.sort_values('balanced_accuracy', ascending=False))

Meta learner best parameters: {'C': 10}


,model,balanced_accuracy,accuracy,f1,roc_auc,best_cv_balanced_accuracy
2,OOF stacked model,0.823942,0.806633,0.679668,0.910452,0.826339
1,Tuned Random Forest,0.822606,0.801924,0.675716,0.910394,0.826228
0,Tuned Logistic Regression,0.812451,0.791381,0.661799,0.897959,0.816971


## Evaluation

The engineered data started with 632 usable features after one-hot encoding, numeric polynomial expansion, and numeric-by-category interactions. Reducing the set to 20 features did lower the untuned baseline scores a little, logistic regression went from 0.8243 balanced accuracy on the large feature set to 0.8125 on the selected 20, and the random forest went from 0.7815 to 0.7778. That is a reasonable tradeoff because the selected model uses far fewer variables and is easier to interpret.

Tuning had a clear effect on the random forest. The selected-feature random forest baseline scored 0.7778 balanced accuracy, while the tuned random forest improved to 0.8226 on the test set. Logistic regression was already close to its best setting: the tuned model scored 0.8125 balanced accuracy, about the same as the selected-feature baseline, but the grid search still showed that an L1 model with C=3 was preferred over the default-style L2 setup.

The stacked model used out-of-fold probabilities from the tuned logistic regression and tuned random forest. It produced the best test balanced accuracy in my run at 0.8239, slightly above the tuned random forest at 0.8226 and above logistic regression at 0.8125. The improvement is small, but it shows the meta learner was able to use the two base learners together without training on in-fold predictions.

The consistently important features were marital_status_Married_civ_spouse, educational_num_z, capital_gain_z, hours_per_week_z, age_z, and relationship or occupation indicators such as relationship_Own_child, relationship_Wife, occupation_Exec_managerial, and occupation_Other_service. The strongest engineered interactions were age or hours worked crossed with marital status, which makes sense because income patterns change with age, work time, and household status.

Removed features included fnlwgt, rare or weaker country indicators, and hundreds of numeric-by-category interactions that ranked below the final top 20. I removed them because the combined ranking tools did not show enough ANOVA, L1, or tree-based signal to justify keeping them in such a small final feature set.